# 🎙️ Jarvis — Voice Assistant

Speak to Jarvis using your microphone. Jarvis will:
1. Listen to you via browser mic
2. Transcribe using OpenAI Whisper
3. Think using GPT-4o
4. Speak the response back

> **Requires:** Run `01_setup.ipynb` first and set your `OPENAI_API_KEY` in Colab Secrets.

In [ ]:
# Load API key and imports
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

import openai
import whisper
import base64
import tempfile
from gtts import gTTS
from IPython.display import Javascript, Audio, display
from google.colab import output

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
whisper_model = whisper.load_model('base')  # Options: tiny, base, small, medium
print('Jarvis voice assistant ready.')

In [ ]:
# Browser mic recorder (runs JavaScript in your browser)
RECORD_JS = """
async function recordAudio(durationMs) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await new Promise(r => setTimeout(r, durationMs));
  recorder.stop();
  await new Promise(r => recorder.onstop = r);
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks, { type: 'audio/webm' });
  const reader = new FileReader();
  return new Promise(resolve => {
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}
"""

def record_from_browser(seconds=5):
    """Record audio from browser mic and save to file."""
    display(Javascript(RECORD_JS))
    print(f'Recording for {seconds} seconds... Speak now!')
    data_url = output.eval_js(f'recordAudio({seconds * 1000})')
    audio_bytes = base64.b64decode(data_url.split(',')[1])
    path = '/tmp/jarvis_input.webm'
    with open(path, 'wb') as f:
        f.write(audio_bytes)
    print('Recording saved.')
    return path

print('Recorder ready.')

In [ ]:
# Transcribe audio with Whisper
def transcribe(audio_path):
    result = whisper_model.transcribe(audio_path)
    text = result['text'].strip()
    print(f'You said: "{text}"')
    return text

print('Transcription function ready.')

In [ ]:
# Jarvis brain — GPT-4o
conversation_history = [
    {
        'role': 'system',
        'content': (
            'You are Jarvis, a highly intelligent personal AI assistant. '
            'Respond concisely, clearly, and helpfully. '
            'You have a calm, professional tone like the AI from Iron Man. '
            'Keep responses under 3 sentences unless a detailed answer is needed.'
        )
    }
]

def ask_jarvis(user_text):
    conversation_history.append({'role': 'user', 'content': user_text})
    response = client.chat.completions.create(
        model='gpt-4o',
        messages=conversation_history,
        max_tokens=300
    )
    reply = response.choices[0].message.content.strip()
    conversation_history.append({'role': 'assistant', 'content': reply})
    print(f'Jarvis: "{reply}"')
    return reply

print('GPT-4o brain ready.')

In [ ]:
# Text-to-speech using gTTS + play in Colab
def speak(text):
    tts = gTTS(text=text, lang='en', slow=False)
    path = '/tmp/jarvis_response.mp3'
    tts.save(path)
    display(Audio(path, autoplay=True))

print('Voice output ready.')

In [ ]:
# ▶️ TALK TO JARVIS — Run this cell each time you want to speak
# Change seconds= to give yourself more time to speak

audio_file = record_from_browser(seconds=6)
user_text = transcribe(audio_file)

if user_text:
    reply = ask_jarvis(user_text)
    speak(reply)
else:
    print('Nothing heard. Try again.')

In [ ]:
# ✍️ TYPE to Jarvis instead of speaking (fallback)
user_input = input('Type your message to Jarvis: ')
reply = ask_jarvis(user_input)
speak(reply)